### Ingestion del archivo "language.csv"

In [0]:
dbutils.widgets.text("param_file_date","2014-12-16")
v_file_date = dbutils.widgets.get("param_file_date")

In [0]:
dbutils.widgets.text("parm_enviroment","")
v_enviroment = dbutils.widgets.get("parm_enviroment")

In [0]:
%run "../../utilerias/configurationes"

#### Paso 1- Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_schema = StructType([
  StructField('languageId', IntegerType(), True),
  StructField('languageCode', StringType(), True),
  StructField('languageName', StringType(), True)
]) 

In [0]:
languages_df = spark.read.option("header", True).schema(language_schema).csv(f"{raw_folder_path}/{v_file_date}/language.csv")

#### Paso 2 - Agregar las columnas "ingestion_date" y "enviroment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
language_df_final = languages_df.withColumns({"ingestion_date": current_timestamp(), "enviroment": lit(v_enviroment),\
    "file_date": lit(v_file_date)}) 

#### Paso 3 - Escribir datos en una tabla Delta. 

In [0]:
language_df_final.write.mode("overwrite").format("delta").saveAsTable("smartdata_proyect_bronze.languages")

In [0]:
%sql
select * from smartdata_proyect_bronze.languages

In [0]:
dbutils.notebook.exit("success")